# Commune PC1 vectors from household food consumption

Mean-centered singular value decomposition of the per-household food
consumption matrix in `data/mira_cleaned_git.csv`. The leading component
(DC1) is projected back onto every household record and averaged to
commune-month, producing the `PC1_Score` column released in
`data/commune_monthly.csv`.

Communes are numbered 1-6 as in the manuscript; `data/commune_lookup.csv`
records each commune's name and its code in the source shapefile.

All paths are relative to this notebook, so the notebook runs as-is from
the `code/` directory of a clone.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns; sns.set()

DATA = "../data"

In [ ]:
# ---------------------------------------------------------------------
# Load the de-identified household records and the commune lookup table
# ---------------------------------------------------------------------
mira = pd.read_csv(f"{DATA}/mira_cleaned_git.csv")
commune_lookup = pd.read_csv(f"{DATA}/commune_lookup.csv")
commune_name = dict(zip(commune_lookup["commune"], commune_lookup["commune_name"]))

print(f"{len(mira):,} household records, communes {sorted(mira['commune'].unique())}")
print(commune_lookup)

In [ ]:
# ---------------------------------------------------------------------
# Food items entering the decomposition
# ---------------------------------------------------------------------
food_columns = ['Maize', 'Sorghum', 'Millet', 'Rice', 'Cassava Root', 'Cassava Leaves', 'Sweet Potato',
                'Other Tubers', 'Peanuts', 'Cowpeas', 'Beans', 'Dolique', 'Other Legumues', 'green/white cactus',
                'red cactus', 'vegetables', 'melons', 'other fruit', 'cactus leaves', 'other leaves',
                'watermelon', 'peas']

food_labels = ['Maize', 'Sorghum', 'Millet', 'Rice', 'Cassava Root', 'Cassava Leaves', 'Sweet Potato',
               'Other Tubers', 'Peanuts', 'Cowpeas', 'Beans', 'Dolique', 'Other Legumes', 'Green/White Cactus',
               'Red Cactus', 'Vegetables', 'Melons', 'Other Fruit', 'Cactus Leaves', 'Other Leaves',
               'Watermelon', 'Peas']

# Numeric, with missing consumption filled by the item mean
foods = mira[food_columns].apply(pd.to_numeric, errors="coerce")
foods = foods.fillna(foods.mean())

# Mean centering: subtract each item's mean across all households
foods_centered = foods - foods.mean()

print(foods_centered.shape)

In [ ]:
# ---------------------------------------------------------------------
# SVD of the mean-centered household x food matrix
# ---------------------------------------------------------------------
U, Sigma, VT = np.linalg.svd(foods_centered, full_matrices=False)

variance_explained = Sigma**2 / np.sum(Sigma**2)
for k in range(3):
    print(f"PC{k+1}: {variance_explained[k]*100:.2f}% of variance")

In [ ]:
# ---------------------------------------------------------------------
# Component loadings
# ---------------------------------------------------------------------
sns.set_style("white", {"axes.edgecolor": "black"})

# PC1 - the manuscript figure
plt.figure(figsize=[16, 6])
plt.bar(food_labels, VT[0, :], color="#103F60")
plt.gca().set_facecolor("white")
plt.xticks(rotation=90)
plt.ylabel("Loading")
plt.grid(False)
plt.rcParams.update({"font.size": 24, "figure.facecolor": "white"})
plt.savefig("first_principal_component_barplot.jpg", format="jpg", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()

# PC2 and PC3, for reference
for k in (1, 2):
    plt.figure(figsize=[14, 4])
    plt.bar(food_labels, VT[k, :])
    plt.title(f"Principal Component {k+1} (mean-centered data)")
    plt.xticks(rotation=90)
    plt.ylabel("Loading")
    plt.grid(False)
    plt.show()

In [ ]:
# ---------------------------------------------------------------------
# Project the leading component back onto every household record
# ---------------------------------------------------------------------
mira["PC1_Score"] = foods_centered @ VT[0, :]

# Undated records (blank year/month/day) cannot enter a monthly series
dated = mira.dropna(subset=["year", "month"]).copy()
print(f"{len(mira) - len(dated):,} undated records dropped ({(1-len(dated)/len(mira))*100:.1f}%)")

dated["YearMonth"] = (dated["year"].astype(int).astype(str) + "-" +
                      dated["month"].astype(int).astype(str).str.zfill(2))

commune_pc1 = (
    dated.groupby(["commune", "YearMonth"], as_index=False)["PC1_Score"].mean()
         .sort_values(["commune", "YearMonth"])
         .reset_index(drop=True)
)

commune_pc1["commune_name"] = commune_pc1["commune"].map(commune_name)
print(commune_pc1.head())
print(commune_pc1.groupby("commune").size())

In [ ]:
# ---------------------------------------------------------------------
# Write the commune-month PC1 vectors
# ---------------------------------------------------------------------
# data/commune_monthly.csv is this table joined to commune-mean NDVI; the NDVI
# series are extracted from the satellite record outside this repository.
out = f"{DATA}/commune_pc1_monthly.csv"
commune_pc1.to_csv(out, index=False)
print("wrote", out)

# Agreement with the released file
released = pd.read_csv(f"{DATA}/commune_monthly.csv")
check = released.merge(commune_pc1, on=["commune", "YearMonth"], suffixes=("_released", "_here"))
print(f"{len(check)} commune-months matched; "
      f"max abs difference = {(check['PC1_Score_released'] - check['PC1_Score_here']).abs().max():.2e}")